In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from datetime import datetime, timedelta
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# 确保图形在Jupyter Notebook中显示
%matplotlib inline

In [ ]:
import pandas as pd
import numpy as np
# 读取CSV文件
df_apple1 = pd.read_csv("688981_posts_data.csv")
df_apple2 = pd.read_csv("000524_posts_data.csv")
df_apple3 = pd.read_csv("002475_posts_data.csv")

In [ ]:
# 所有时间都是 2025 年的
df_apple1['Date'] = "2025-" + df_apple1['Date']
df_apple1['Date'] = pd.to_datetime(df_apple1['Date']).dt.date
df_apple2['Date'] = "2025-" + df_apple2['Date']
df_apple2['Date'] = pd.to_datetime(df_apple2['Date']).dt.date
df_apple3['Date'] = "2025-" + df_apple3['Date']
df_apple3['Date'] = pd.to_datetime(df_apple3['Date']).dt.date
# df_apple['Date'] = pd.to_datetime(df_apple['Date'], format="%Y-%m-%d %H:%M")
df_apple1.head()
df_apple2.head()
df_apple3.head()

In [ ]:
import re
# 简单清洗文本（去除特殊符号、空格等）
def clean_text(text):
    text = re.sub(r'[^\w\s]', '', str(text))  # 去除非字母数字字符
    text = text.strip()  # 去除首尾空格
    return text
df_apple1["title"] = df_apple1["title"].apply(clean_text)
df_apple2["title"] = df_apple2["title"].apply(clean_text)
df_apple3["title"] = df_apple3["title"].apply(clean_text)
print(df_apple1.head())  # 检查清洗后的数据
print(df_apple2.head())  # 检查清洗后的数据
print(df_apple3.head())  # 检查清洗后的数据

In [ ]:
# 按照股票代码合并数据, 加上股票代码一列
df_apple1['ticker'] = '688981'
df_apple2['ticker'] = '000524'
df_apple3['ticker'] = '002475'
df_apple = pd.concat([df_apple1, df_apple2, df_apple3], ignore_index=True)
df_apple = df_apple[['ticker', 'Date', 'title', 'read_count']]
df_apple['read_count'] = df_apple['read_count'].astype(int)
df_apple.head()


In [ ]:
# 情感分析
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
# 使用专为中文设计的情感分析模型
model_name = "yiyanghkust/finbert-tone-chinese"  # 金融情感分析模型
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
# 创建中文情感分析pipeline
sentiment_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)
# 分析中文文本
results_apple = []
for tweet in df_apple["title"]:
    try:
        sentiment = sentiment_pipeline(tweet)[0]
        # 转换标签格式以匹配后续处理代码
        # 这个模型的标签是 "positive" 和 "negative"
        # 股票代码也加上
        results_apple.append((tweet, [sentiment], df_apple.loc[df_apple['title'] == tweet, 'ticker'].values[0]))
    except Exception as e:
        # 处理可能的错误，比如文本太长或格式问题
        print(f"处理文本时出错: {tweet}")
        print(e)
        # 股票代码
        results_apple.append((tweet, [{"label": "neutral", "score": 0.5}], df_apple.loc[df_apple['title'] == tweet, 'ticker'].values[0]))
results_apple

In [ ]:
final_scores = []
for result in results_apple:
    sentiment = result[1][0]
    label = sentiment["label"]
    score = sentiment["score"]

    # LABEL_0: Negative
    # LABEL_1: Neutral
    # LABEL_2: Positive
    if label == "Positive":
        final_scores.append(score)
    elif label == "Negative":
        final_scores.append(-1*score)
    else:
        final_scores.append(0)
df_apple.loc[:,"score"] = final_scores
df_apple

In [ ]:
import pandas as pd

# 每日加权平均分，按股票代码分组
# 首先确保日期格式正确
df_apple["Date"] = pd.to_datetime(df_apple["Date"]).dt.date
# 计算加权分数
df_apple["weighted_score"] = df_apple["score"] * df_apple["read_count"]
# 按日期和股票代码分组计算加权平均
daily_avg = df_apple.groupby(["Date", "ticker"]).apply(
    lambda x: x["weighted_score"].sum() / x["read_count"].sum() if x["read_count"].sum() > 0 else 0
).reset_index(name="score")
# 4. 输出结果
print("每日加权平均分：")
print(daily_avg)


In [ ]:
df_apple.describe()

In [ ]:
import akshare as ak
from datetime import datetime, timedelta

# 定义股票列表和数据时间范围（注意：akshare主要支持A股，这里以A股代码为例）
tickers = ['688981', '000524', '002475']  
# 设置开始和结束日期 2.9-5.7
start_date = "20250209"
end_date = "20250507"

# 创建一个空的DataFrame来存储所有股票的数据
all_data = pd.DataFrame()
# 循环获取每只股票的数据
for ticker in tickers:
    # 使用 akshare 获取股票历史数据
    data = ak.stock_zh_a_hist(symbol=ticker, period="daily", start_date=start_date, end_date=end_date)
    # 数据处理
    data = data.rename(columns={
        "日期": "Date",
        "收盘": "close",
    })[["Date", "close"]]  # 只保留这两列
    # 添加股票代码列
    data['ticker'] = ticker
    # 将日期转换为字符串格式
    data['Date'] = pd.to_datetime(data['Date']).dt.strftime('%Y-%m-%d')
    # 合并数据
    all_data = pd.concat([all_data, data], ignore_index=True)
# 将日期转换为datetime格式
all_data['Date'] = pd.to_datetime(all_data['Date'])
# 打印合并后的数据
print(all_data.head())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from matplotlib.font_manager import FontProperties

# 设置中文字体
font = FontProperties(fname='C:/Windows/Fonts/simhei.ttf')  # Windows中文黑体

# 将值数组转换为数据集矩阵
def create_dataset(dataset, look_back=1):
    dataX, dataY = [], []
    for i in range(len(dataset)-look_back-1):
        dataX.append(dataset[i:(i+look_back), :])
        dataY.append(dataset[i + look_back, 0])  # 我们只预测收盘价
    return np.array(dataX), np.array(dataY)

# 设置随机种子以便结果可复现
np.random.seed(42)
tf.random.set_seed(42)

# 将原始daily_avg数据按ticker分组
tickers = daily_avg['ticker'].unique()
print(f"处理的股票代码: {tickers}")

# 创建一个大的图表来显示所有股票的预测结果
plt.figure(figsize=(18, 18))

# 为每个股票代码分别创建模型和预测
for i, ticker in enumerate(tickers):
    print(f"\n处理股票 {ticker}...")
    
    # 筛选当前股票的数据
    ticker_daily_avg = daily_avg[daily_avg['ticker'] == ticker].copy()
    
    # 获取对应的股价数据
    ticker_data = all_data[all_data['ticker'] == ticker]
    # print(ticker_data.head())

    # 处理日期格式
    ticker_daily_avg['Date'] = pd.to_datetime(ticker_daily_avg['Date'])
    ticker_data['Date'] = pd.to_datetime(ticker_data['Date'])
    
    # 补全缺失日期,以data的日期为准
    ticker_daily_avg = ticker_daily_avg.set_index('Date').reindex(ticker_data['Date']).reset_index()
    
    # 对缺失score的日期插值
    ticker_daily_avg['score'] = ticker_daily_avg['score'].interpolate(method='linear')
    ticker_daily_avg['score'] = ticker_daily_avg['score'].fillna(0)  # 填充可能的缺失值
    
    # 将日期转换为字符串格式
    ticker_daily_avg['Date'] = ticker_daily_avg['Date'].dt.strftime('%Y-%m-%d')
    ticker_data['Date'] = ticker_data['Date'].dt.strftime('%Y-%m-%d')
    
    # 合并当前股票的情感数据和价格数据
    ticker_merged_data = pd.merge(ticker_daily_avg, ticker_data[['Date', 'close']], on='Date')
    ticker_merged_data.set_index('Date', inplace=True)
    
    # 添加情感特征工程
    ticker_merged_data['score_lag1'] = ticker_merged_data['score'].shift(1)  # 前一天的情绪分数
    ticker_merged_data['score_ma3'] = ticker_merged_data['score'].rolling(window=3).mean()  # 3天平均情绪
    ticker_merged_data['score_ma5'] = ticker_merged_data['score'].rolling(window=5).mean()  # 5天平均情绪
    
    # 删除NaN值
    ticker_merged_data = ticker_merged_data.dropna()
    
    # 如果数据太少，则跳过
    if len(ticker_merged_data) < 10:
        print(f"警告: 股票 {ticker} 数据不足，跳过")
        continue
    
    # 选择特征 - 收盘价和所有情绪相关特征
    features = ['close', 'score_ma5']
    dataset = ticker_merged_data[features].values.astype('float32')
    
    # 规范化数据集
    scaler = MinMaxScaler(feature_range=(0, 1))
    dataset_scaled = scaler.fit_transform(dataset)
    
    # 分割为训练集和测试集
    train_size = int(len(dataset_scaled) * 0.67)
    test_size = len(dataset_scaled) - train_size
    train, test = dataset_scaled[0:train_size,:], dataset_scaled[train_size:len(dataset_scaled),:]
    
    # 重新整形为 X=t 和 Y=t+1
    look_back = 3
    trainX, trainY = create_dataset(train, look_back)
    testX, testY = create_dataset(test, look_back)
    
    # 创建并拟合LSTM网络
    model = Sequential()
    model.add(LSTM(50, input_shape=(look_back, len(features))))
    model.add(Dense(1))
    model.compile(loss='mean_squared_error', optimizer='adam')
    model.fit(trainX, trainY, epochs=100, batch_size=16, verbose=0)
    
    # 进行预测
    trainPredict = model.predict(trainX)
    testPredict = model.predict(testX)
    
    # 创建占位数组用于反归一化
    trainY_reshaped = np.zeros((len(trainY), len(features)))
    trainY_reshaped[:,0] = trainY
    trainY_reshaped[:,1] = train[:len(trainY), 1]
    
    testY_reshaped = np.zeros((len(testY), len(features)))
    testY_reshaped[:,0] = testY
    testY_reshaped[:,1] = test[:len(testY), 1]
    
    trainPredict_reshaped = np.zeros((len(trainPredict), len(features)))
    trainPredict_reshaped[:,0] = trainPredict[:,0]
    trainPredict_reshaped[:,1] = train[:len(trainPredict), 1]
    
    testPredict_reshaped = np.zeros((len(testPredict), len(features)))
    testPredict_reshaped[:,0] = testPredict[:,0]
    testPredict_reshaped[:,1] = test[:len(testPredict), 1]
    
    # 反归一化
    trainPredict = scaler.inverse_transform(trainPredict_reshaped)[:,0]
    trainY = scaler.inverse_transform(trainY_reshaped)[:,0]
    testPredict = scaler.inverse_transform(testPredict_reshaped)[:,0]
    testY = scaler.inverse_transform(testY_reshaped)[:,0]
    
    # 计算均方根误差
    trainScore = np.sqrt(mean_squared_error(trainY, trainPredict))
    testScore = np.sqrt(mean_squared_error(testY, testPredict))
    print(f'{ticker} 训练集得分: {trainScore:.2f} RMSE')
    print(f'{ticker} 测试集得分: {testScore:.2f} RMSE')
    
    # 准备绘图数据
    dataset_inv = scaler.inverse_transform(dataset_scaled)
    actual_close = dataset_inv[:,0]
    
    # 创建预测结果的完整序列
    trainPredictPlot = np.empty_like(actual_close)
    trainPredictPlot[:] = np.nan
    trainPredictPlot[look_back:look_back+len(trainPredict)] = trainPredict
    
    testPredictPlot = np.empty_like(actual_close)
    testPredictPlot[:] = np.nan
    testPredictPlot[len(trainPredict)+(look_back*2)+1:len(actual_close)-1] = testPredict
    
    # 在子图中绘制当前股票的预测结果
    plt.subplot(len(tickers), 1, i+1)
    plt.plot(actual_close, label='实际价格')
    plt.plot(trainPredictPlot, label='训练集预测')
    plt.plot(testPredictPlot, label='测试集预测')
    plt.title(f'股票 {ticker} 价格预测', fontproperties=font)
    plt.xlabel('时间', fontproperties=font)
    plt.ylabel('价格', fontproperties=font)
    plt.legend(prop=font)

plt.tight_layout()
plt.show()

# 可选：保存每个股票的预测结果到csv文件
for ticker in tickers:
    ticker_daily_avg = daily_avg[daily_avg['ticker'] == ticker]
    if not ticker_daily_avg.empty:
        ticker_daily_avg.to_csv(f"{ticker}_predictions.csv", index=False)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from matplotlib.font_manager import FontProperties
from tensorflow.keras.models import load_model  # 可选：如果需要保存/加载模型

# 设置中文字体
font = FontProperties(fname='C:/Windows/Fonts/simhei.ttf')  # Windows中文黑体

# 1. 使用LSTM模型的预测结果计算预期收益率
# ===================================================

# 存储每只股票的预测结果和预测收益率
stock_predictions = {}
stock_predicted_returns = {}

# 为每只股票处理预测数据
for ticker in tickers:
    print(f"\n准备股票 {ticker} 的预测收益率...")
    
    # 筛选当前股票的数据
    ticker_data = all_data[all_data['ticker'] == ticker]
    ticker_daily_avg = daily_avg[daily_avg['ticker'] == ticker]
    
    if not ticker_data.empty and not ticker_daily_avg.empty:
        # 准备和合并数据，与先前的LSTM预测代码相同
        ticker_data['Date'] = pd.to_datetime(ticker_data['Date'])
        ticker_daily_avg['Date'] = pd.to_datetime(ticker_daily_avg['Date'])
        
        # 合并价格和情感数据
        ticker_merged_data = pd.merge(ticker_data, ticker_daily_avg, on='Date', how='inner')
        ticker_merged_data.set_index('Date', inplace=True)
        
        # 添加情感特征工程
        ticker_merged_data['score_lag1'] = ticker_merged_data['score'].shift(1)
        ticker_merged_data['score_ma3'] = ticker_merged_data['score'].rolling(window=3).mean()
        ticker_merged_data['score_ma5'] = ticker_merged_data['score'].rolling(window=5).mean()
        ticker_merged_data = ticker_merged_data.dropna()
        
        if len(ticker_merged_data) < 10:
            print(f"警告: 股票 {ticker} 数据不足，使用默认预期收益率")
            # 使用默认预期收益率
            stock_predicted_returns[ticker] = pd.Series(0.0005, index=pd.date_range(start='2025-01-01', periods=10))
            continue
        
        # 选择特征并进行预处理
        features = ['close', 'score_ma5']
        dataset = ticker_merged_data[features].values.astype('float32')
        scaler = MinMaxScaler(feature_range=(0, 1))
        dataset_scaled = scaler.fit_transform(dataset)
        
        # 分割为训练集和测试集
        train_size = int(len(dataset_scaled) * 0.50)
        test_size = len(dataset_scaled) - train_size
        train, test = dataset_scaled[0:train_size,:], dataset_scaled[train_size:len(dataset_scaled),:]
        
        # 重新整形为 X=t 和 Y=t+1
        look_back = 3
        trainX, trainY = create_dataset(train, look_back)
        testX, testY = create_dataset(test, look_back)
        
        # 创建并拟合LSTM网络 (或加载已保存的模型)
        model = Sequential()
        model.add(LSTM(50, input_shape=(look_back, len(features))))
        model.add(Dense(1))
        model.compile(loss='mean_squared_error', optimizer='adam')
        model.fit(trainX, trainY, epochs=100, batch_size=16, verbose=0)
        
        # 进行预测
        trainPredict = model.predict(trainX)
        testPredict = model.predict(testX, verbose=0)
        
        # 反归一化处理
        testPredict_reshaped = np.zeros((len(testPredict), len(features)))
        testPredict_reshaped[:,0] = testPredict[:,0]
        testPredict_reshaped[:,1] = test[:len(testPredict), 1]
        
        testY_reshaped = np.zeros((len(testY), len(features)))
        testY_reshaped[:,0] = testY
        testY_reshaped[:,1] = test[:len(testY), 1]
        
        # 反归一化
        testPredict_prices = scaler.inverse_transform(testPredict_reshaped)[:,0]
        testY_prices = scaler.inverse_transform(testY_reshaped)[:,0]
        
        # 存储预测结果
        stock_predictions[ticker] = testPredict_prices
        
        # 2. 计算预测收益率: 根据预测价格变动计算
        # 使用(第二天价格/第一天价格)-1作为收益率
        predicted_returns = np.zeros(len(testPredict_prices)-1)
        for i in range(len(testPredict_prices)-1):
            predicted_returns[i] = (testPredict_prices[i+1]/testPredict_prices[i]) - 1
        
        # 将预测收益率转换为pandas Series以便后续使用
        # 假设测试集的日期从ticker_merged_data的后三分之一开始
        test_dates = ticker_merged_data.index[train_size+look_back+1:train_size+look_back+1+len(predicted_returns)]
        stock_predicted_returns[ticker] = pd.Series(predicted_returns, index=test_dates)
        
        print(f"股票 {ticker} 的平均预测收益率: {predicted_returns.mean():.4f}")

# 3. 创建预期收益率数据框
# ===================================================

# 创建预测收益率数据框 - 使用相同的日期索引合并所有股票
predicted_returns_df = pd.DataFrame()

for ticker in tickers:
    if ticker in stock_predicted_returns:
        predicted_returns_df[ticker] = stock_predicted_returns[ticker]

# 删除缺失值并确保所有股票在相同日期有数据
predicted_returns_df = predicted_returns_df.dropna()

# 4. 计算协方差矩阵
# ===================================================

# 计算基于预测收益率的协方差矩阵
# 这里我们可以选择使用预测收益率自身的协方差或使用实际历史收益率的协方差
if len(predicted_returns_df) >= 10:  # 如果有足够多的预测数据点
    print("使用预测收益率计算协方差矩阵")
    cov_matrix = predicted_returns_df.cov()
else:
    print("预测数据点不足，使用实际历史收益率计算协方差矩阵")
    # 计算实际历史收益率
    returns_df = pd.DataFrame()
    for ticker in tickers:
        ticker_data = all_data[all_data['ticker'] == ticker].copy()
        if len(ticker_data) > 0:
            ticker_data['Date'] = pd.to_datetime(ticker_data['Date'])
            ticker_data.set_index('Date', inplace=True)
            ticker_data['return'] = ticker_data['close'].pct_change()
            returns_df[ticker] = ticker_data['return']
    
    returns_df = returns_df.dropna()
    cov_matrix = returns_df.cov()

# 5. 准备预期收益率
# ===================================================

# 使用预测收益率的均值作为预期收益率
expected_returns = {ticker: predicted_returns_df[ticker].mean() for ticker in predicted_returns_df.columns}

# 打印预期收益率以便验证
print("\n各股票预期日收益率:")
for ticker, exp_return in expected_returns.items():
    print(f"{ticker}: {exp_return*100:.4f}%")

# 6. 投资组合优化函数
# ===================================================

# 计算投资组合的预期收益率
def portfolio_expected_return(weights, returns):
    return np.sum(weights * np.array(list(returns.values())))

# 计算投资组合的波动率（风险）
def portfolio_volatility(weights, cov_matrix):
    variance = np.dot(weights.T, np.dot(cov_matrix, weights))
    # Ensure variance is not negative due to precision errors
    return np.sqrt(np.maximum(0, variance))

# 计算夏普比率 (风险调整后的收益)
def sharpe_ratio(weights, returns, cov_matrix, risk_free_rate=0.01):
    return (portfolio_expected_return(weights, returns) - risk_free_rate) / portfolio_volatility(weights, cov_matrix)

# 目标函数: 最大化夏普比率
def negative_sharpe_ratio(weights, returns, cov_matrix, risk_free_rate=0.01):
    return -sharpe_ratio(weights, returns, cov_matrix, risk_free_rate)

# 约束条件: 权重总和为1
def weight_sum_constraint(weights):
    return np.sum(weights) - 1

# 7. 投资组合优化
# ===================================================

# 初始权重 (等比例分配)
n_assets = len(tickers)
initial_weights = np.ones(n_assets) / n_assets

# 优化求解 - 最大夏普比率投资组合
bounds = tuple((0.05, 0.6) for _ in range(n_assets))
constraints = {'type': 'eq', 'fun': weight_sum_constraint}

optimal_sharpe_result = minimize(
    negative_sharpe_ratio, 
    initial_weights, 
    args=(expected_returns, cov_matrix),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

optimal_sharpe_weights = optimal_sharpe_result['x']

# 最小波动率投资组合
def min_volatility(weights, cov_matrix):
    return portfolio_volatility(weights, cov_matrix)

min_vol_result = minimize(
    min_volatility,
    initial_weights,
    args=(cov_matrix,),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

min_vol_weights = min_vol_result['x']

# 8. 显示投资组合优化结果
# ===================================================

print("\n====== 投资组合优化结果（基于LSTM预测）======")

print("\n最优夏普比率投资组合 (最佳风险收益比):")
for i, ticker in enumerate(tickers):
    print(f"{ticker}: {optimal_sharpe_weights[i]*100:.2f}%")

expected_return = portfolio_expected_return(optimal_sharpe_weights, expected_returns)
expected_volatility = portfolio_volatility(optimal_sharpe_weights, cov_matrix)
expected_sharpe = sharpe_ratio(optimal_sharpe_weights, expected_returns, cov_matrix)

print(f"预期每日收益率: {expected_return*100:.4f}%")
print(f"预期每日波动率: {expected_volatility*100:.4f}%")
print(f"预期年化收益率: {expected_return*252*100:.2f}%")
print(f"预期年化波动率: {expected_volatility*np.sqrt(252)*100:.2f}%")
print(f"夏普比率: {expected_sharpe:.4f}")

print("\n最小波动率投资组合 (最低风险):")
for i, ticker in enumerate(tickers):
    print(f"{ticker}: {min_vol_weights[i]*100:.2f}%")

min_vol_return = portfolio_expected_return(min_vol_weights, expected_returns)
min_vol_volatility = portfolio_volatility(min_vol_weights, cov_matrix)
min_vol_sharpe = sharpe_ratio(min_vol_weights, expected_returns, cov_matrix)

print(f"预期每日收益率: {min_vol_return*100:.4f}%")
print(f"预期每日波动率: {min_vol_volatility*100:.4f}%")
print(f"预期年化收益率: {min_vol_return*252*100:.2f}%")
print(f"预期年化波动率: {min_vol_volatility*np.sqrt(252)*100:.2f}%")
print(f"夏普比率: {min_vol_sharpe:.4f}")

# 9. 可视化投资组合权重
# ===================================================

plt.figure(figsize=(15, 10))

# 最优夏普比率投资组合
plt.subplot(2, 1, 1)
plt.pie(optimal_sharpe_weights, labels=tickers, autopct='%1.1f%%', startangle=90)
plt.title('基于LSTM预测的最优夏普比率投资组合', fontproperties=font, fontsize=14)

# 最小波动率投资组合
plt.subplot(2, 1, 2)
plt.pie(min_vol_weights, labels=tickers, autopct='%1.1f%%', startangle=90)
plt.title('基于LSTM预测的最小波动率投资组合', fontproperties=font, fontsize=14)

plt.tight_layout()
plt.show()

# 10. 根据风险偏好推荐投资组合
# ===================================================

def recommend_portfolio(risk_score, expected_returns, cov_matrix, tickers):
    """根据风险评分推荐投资组合"""
    n_assets = len(tickers)
    initial_weights = np.ones(n_assets) / n_assets
    
    # 根据风险偏好设置不同的权重约束和优化目标
    if risk_score <= 20:  # 极低风险
        bounds = tuple((0.15, 0.40) for _ in range(n_assets))
        title = "极低风险投资组合"
        objective = min_volatility  # 目标是最小化波动率
        objective_args = (cov_matrix,)
        explanation = "此投资组合追求最低的风险，适合风险承受能力极低的投资者"
    elif risk_score <= 40:  # 低风险
        bounds = tuple((0.10, 0.50) for _ in range(n_assets))
        title = "低风险投资组合"
        objective = min_volatility  # 目标仍是最小化波动率，但限制更宽松
        objective_args = (cov_matrix,)
        explanation = "此投资组合追求较低风险，但允许更大的股票配置灵活性"
    elif risk_score <= 60:  # 中等风险
        bounds = tuple((0.05, 0.60) for _ in range(n_assets))
        title = "平衡型投资组合"
        objective = negative_sharpe_ratio  # 目标是优化风险回报比
        objective_args = (expected_returns, cov_matrix)
        explanation = "此投资组合平衡风险和回报，适合大多数投资者"
    elif risk_score <= 80:  # 高风险
        bounds = tuple((0.05, 0.70) for _ in range(n_assets))
        title = "进取型投资组合"
        # 调整负夏普比率，更偏向收益
        objective = lambda w, er, cov: -sharpe_ratio(w, er, cov) * 0.8 - portfolio_expected_return(w, er) * 0.2
        objective_args = (expected_returns, cov_matrix)
        explanation = "此投资组合偏重回报，承担较高风险，适合进取型投资者"
    else:  # 极高风险
        bounds = tuple((0.05, 0.80) for _ in range(n_assets))
        title = "激进型投资组合"
        # 更强调收益而非夏普比率
        objective = lambda w, er, cov: -portfolio_expected_return(w, er)
        objective_args = (expected_returns, cov_matrix)
        explanation = "此投资组合追求最高回报，愿意承担显著风险，适合激进型投资者"
    
    constraints = {'type': 'eq', 'fun': weight_sum_constraint}
    
    result = minimize(
        objective,
        initial_weights,
        args=objective_args,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )
    
    weights = result['x']
    expected_return = portfolio_expected_return(weights, expected_returns)
    expected_volatility = portfolio_volatility(weights, cov_matrix)
    expected_sharpe = sharpe_ratio(weights, expected_returns, cov_matrix)
    
    return weights, title, explanation, expected_return, expected_volatility, expected_sharpe

# 提示用户输入风险评分
risk_score = int(input("请输入您的风险偏好评分(0-100)，0为最保守，100为最激进："))

# 根据风险评分生成推荐投资组合
rec_weights, rec_title, rec_explanation, rec_return, rec_volatility, rec_sharpe = recommend_portfolio(
    risk_score, expected_returns, cov_matrix, tickers
)

# 显示推荐投资组合
print(f"\n====== {rec_title} (基于LSTM预测) ======")
print(rec_explanation)
print("\n推荐投资组合权重:")
for i, ticker in enumerate(tickers):
    print(f"{ticker}: {rec_weights[i]*100:.2f}%")

print(f"预期每日收益率: {rec_return*100:.4f}%")
print(f"预期每日波动率: {rec_volatility*100:.4f}%")
print(f"预期年化收益率: {rec_return*252*100:.2f}%")
print(f"预期年化波动率: {rec_volatility*np.sqrt(252)*100:.2f}%")
print(f"夏普比率: {rec_sharpe:.4f}")

# 可视化推荐投资组合
plt.figure(figsize=(10, 10))
plt.pie(rec_weights, labels=tickers, autopct='%1.1f%%', startangle=90)
plt.title(f"{rec_title} (基于LSTM预测)", fontproperties=font, fontsize=16)
plt.show()